In [1]:
# =========================
# Industry-only baseline
# Use only broad sector controls
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# =========================
# 1. Load Y
# =========================
Y_PATH = "deepseek_Y_quantile_final.csv"
y = pd.read_csv(Y_PATH)

print("y shape:", y.shape)
print("\ny columns:")
print(y.columns.tolist())


# =========================
# 2. Keep label + SIC description
# =========================
y_keep = y[["ticker", "label", "sic_description"]].drop_duplicates().copy()

label_map = {
    "Vulnerable": 1,
    "Resilient": 0
}
y_keep["label_binary"] = y_keep["label"].map(label_map)

print("\nY label counts:")
print(y_keep["label"].value_counts(dropna=False))


# =========================
# 3. Create broad industry categories
# =========================
def map_broad_sector(s):
    if pd.isna(s):
        return "other"

    s = str(s).lower()

    # Semiconductor / chips / electronics
    if any(k in s for k in [
        "semiconductor", "electronic component", "electronics",
        "chip", "integrated circuit"
    ]):
        return "semiconductor"

    # Software / SaaS / cybersecurity / IT services
    elif any(k in s for k in [
        "software", "prepackaged software", "programming",
        "computer systems design", "data processing",
        "computer integrated systems", "it service", "cyber"
    ]):
        return "software"

    # Internet / platform / media / communications
    elif any(k in s for k in [
        "internet", "online", "portal", "streaming",
        "broadcast", "telecommunication", "communications",
        "advertising"
    ]):
        return "internet"

    # Energy / power / utility
    elif any(k in s for k in [
        "energy", "power", "utility", "utilities",
        "electric", "oil", "gas"
    ]):
        return "energy"

    else:
        return "other"


y_keep["broad_sector"] = y_keep["sic_description"].apply(map_broad_sector)

print("\nBroad sector counts:")
print(y_keep["broad_sector"].value_counts(dropna=False))


# =========================
# 4. Create industry dummies
# =========================
sector_dummies = pd.get_dummies(
    y_keep["broad_sector"],
    prefix="sector",
    drop_first=True,
    dtype=int
)

df_industry = pd.concat([y_keep, sector_dummies], axis=1)

industry_cols = sector_dummies.columns.tolist()

print("\nIndustry control columns:")
print(industry_cols)

print("\nIndustry-only dataset shape:", df_industry.shape)
print("\nDuplicate tickers:", df_industry["ticker"].duplicated().sum())
print("\nMissing values:")
print(df_industry.isna().sum().sort_values(ascending=False).head(20))


# =========================
# 5. Define X and y
# =========================
X = df_industry[industry_cols].copy()
y_binary = df_industry["label_binary"].copy()

print("\nX shape:", X.shape)
print("y shape:", y_binary.shape)


# =========================
# 6. Define models
# =========================
def build_model(model_name):
    if model_name == "logit_l1":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="saga",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "logit_l2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "random_forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "xgboost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                random_state=42,
                eval_metric="logloss"
            ))
        ])

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


# =========================
# 7. Evaluation function
# =========================
def evaluate_model(X, y, model_name):
    model = build_model(model_name)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    return {
        "feature_set": "X_industry_only",
        "model_name": model_name,
        "n_samples": len(X),
        "n_features": X.shape[1],
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }


# =========================
# 8. Run models
# =========================
model_names = ["logit_l1", "logit_l2", "random_forest", "xgboost"]

results = []
for m in model_names:
    print(f"Running {m} ...")
    results.append(evaluate_model(X, y_binary, m))

results_df = pd.DataFrame(results)

print("\n=== Industry-only results ===")
print(results_df.round(4).to_string(index=False))


# =========================
# 9. Save outputs
# =========================
df_industry.to_csv("model_industry_only.csv", index=False)
results_df.to_csv("industry_only_results.csv", index=False)

print("\nSaved files:")
print("  model_industry_only.csv")
print("  industry_only_results.csv")

y shape: (158, 17)

y columns:
['company_id', 'cik', 'ticker', 'company_name', 'sic', 'sic_description', 'start_date_actual', 'start_price', 'end_date_actual', 'end_price', 'raw_return', 'normal_return_spy', 'excess_return', 'CAR', 'MaxDrawdown', 'Volatility', 'label']

Y label counts:
label
Vulnerable    79
Resilient     79
Name: count, dtype: int64

Broad sector counts:
broad_sector
software         62
semiconductor    40
other            39
internet         12
energy            5
Name: count, dtype: int64

Industry control columns:
['sector_internet', 'sector_other', 'sector_semiconductor', 'sector_software']

Industry-only dataset shape: (158, 9)

Duplicate tickers: 0

Missing values:
ticker                  0
label                   0
sic_description         0
label_binary            0
broad_sector            0
sector_internet         0
sector_other            0
sector_semiconductor    0
sector_software         0
dtype: int64

X shape: (158, 4)
y shape: (158,)
Running logit_l1 ...